# 03. Data Preparation — predicción de consumo eléctrico mensual

**Alcance:** carga, validación temporal, creación de `lag_1`, `lag_2`, `lag_3`, target, split cronológico y exportación. No se entrenan modelos ni se ajustan encoders o transformadores.

## 1. Configuración y carga


In [1]:
from __future__ import annotations

from pathlib import Path

import polars as pl
from IPython.display import display

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

CUBE_RELATIVE_PATH = Path("reports") / "eda_outputs" / "resumen_distribuidora_ubigeo_tarifa_periodo.parquet"
CUBE_KEY = ["DISTRIBUIDORA", "UBIGEO", "TARIFA", "PERIODO"]
SERIES_KEY = ["DISTRIBUIDORA", "UBIGEO", "TARIFA"]

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / CUBE_RELATIVE_PATH).is_file():
            return candidate
    raise FileNotFoundError(
        f"No se encontró {CUBE_RELATIVE_PATH} desde {start} ni desde sus directorios padre."
    )

ROOT = find_project_root()
CUBE_PATH = ROOT / CUBE_RELATIVE_PATH
CUBE_SUMMARY_PATH = ROOT / "reports" / "eda_outputs" / "resumen_cubo_ml_corregido.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
MODEL_OUTPUTS_DIR = ROOT / "reports" / "model_outputs"
DATASET_PATH = PROCESSED_DIR / "dataset_ml.parquet"
SUMMARY_PATH = MODEL_OUTPUTS_DIR / "data_preparation_summary.csv"
DIST_SPLIT_PATH = MODEL_OUTPUTS_DIR / "distribucion_split_distribuidora.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {ROOT}")
print(f"Cubo corregido:     {CUBE_PATH}")
print(f"Dataset de salida:  {DATASET_PATH}")

Raíz del proyecto: C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08
Cubo corregido:     C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\reports\eda_outputs\resumen_distribuidora_ubigeo_tarifa_periodo.parquet
Dataset de salida:  C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\data\processed\dataset_ml.parquet


In [2]:
df_raw = pl.read_parquet(CUBE_PATH)

# Adaptación segura del esquema: se aceptan diferencias solo de mayúsculas/minúsculas.
canonical_columns = [
    "DISTRIBUIDORA", "UBIGEO", "TARIFA", "PERIODO", "n_registros",
    "consumo_total_mwh", "importe_total_miles", "consumo_mediano",
]
lookup = {column.lower(): column for column in df_raw.columns}
missing = [column for column in canonical_columns if column.lower() not in lookup]
if missing:
    raise ValueError(f"El cubo corregido no contiene las columnas necesarias: {missing}")

rename_map = {
    lookup[column.lower()]: column
    for column in canonical_columns
    if lookup[column.lower()] != column
}
df = df_raw.rename(rename_map).with_columns(
    pl.col("DISTRIBUIDORA").cast(pl.String).str.to_uppercase(),
    pl.col("UBIGEO").cast(pl.String),
    pl.col("TARIFA").cast(pl.String),
    pl.col("PERIODO").cast(pl.Int32),
    pl.col("consumo_total_mwh").cast(pl.Float64),
)

cube_build_summary = pl.read_csv(CUBE_SUMMARY_PATH)
initial_overview = df.select(
    pl.len().alias("filas"),
    pl.col("DISTRIBUIDORA").n_unique().alias("distribuidoras"),
    pl.col("UBIGEO").n_unique().alias("ubigeos"),
    pl.col("TARIFA").n_unique().alias("tarifas"),
    pl.col("PERIODO").n_unique().alias("periodos"),
    pl.col("PERIODO").min().alias("periodo_min"),
    pl.col("PERIODO").max().alias("periodo_max"),
    pl.col("n_registros").sum().alias("registros_fuente_reconciliados"),
    pl.col("consumo_total_mwh").sum().alias("consumo_total_mwh"),
)

print(f"Dimensiones: {df.shape}")
print(f"Columnas: {df.columns}")
print("Tipos de datos:")
print(df.schema)
print("Primeros registros:")
display(df.head())
print("Resumen inicial:")
display(initial_overview)
print("Validación generada durante la reconstrucción desde data/interim/:")
display(cube_build_summary)

Dimensiones: (44961, 8)
Columnas: ['DISTRIBUIDORA', 'UBIGEO', 'TARIFA', 'PERIODO', 'n_registros', 'consumo_total_mwh', 'importe_total_miles', 'consumo_mediano']
Tipos de datos:
Schema({'DISTRIBUIDORA': String, 'UBIGEO': String, 'TARIFA': String, 'PERIODO': Int32, 'n_registros': UInt32, 'consumo_total_mwh': Float64, 'importe_total_miles': Float32, 'consumo_mediano': Float32})
Primeros registros:


DISTRIBUIDORA,UBIGEO,TARIFA,PERIODO,n_registros,consumo_total_mwh,importe_total_miles,consumo_mediano
str,str,str,i32,u32,f64,f32,f32
"""ENOSA""","""010509""","""BT5B""",202407,1,0.0,0.01,1.0
"""ENOSA""","""010509""","""BT5B""",202408,1,0.0,0.0,0.0
"""ENOSA""","""010509""","""BT5B""",202409,1,0.0,0.01,2.0
"""ENOSA""","""010509""","""BT5B""",202410,1,0.0,0.0,0.0
"""ENOSA""","""010509""","""BT5B""",202411,1,0.0,0.01,0.0


Resumen inicial:


filas,distribuidoras,ubigeos,tarifas,periodos,periodo_min,periodo_max,registros_fuente_reconciliados,consumo_total_mwh
u32,u32,u32,u32,u32,i32,i32,u32,f64
44961,2,770,15,46,202210,202608,51441180,6.6534e6


Validación generada durante la reconstrucción desde data/interim/:


DISTRIBUIDORA,filas_cubo,registros_fuente,ubigeos,tarifas,periodos,periodo_min,periodo_max,consumo_total_mwh
str,i64,i64,i64,i64,i64,i64,i64,f64
"""ENOSA""",16765,18518031,258,14,32,202401,202608,2.6080e6
"""HIDRANDINA""",28196,32923149,615,13,33,202210,202507,4.0454e6


## 2. Validación de la estructura temporal

`PERIODO` se valida como `YYYYMM`, se convierte en una fecha mensual segura y también en un índice de meses. 

El índice permite comprobar continuidad calendario sin confundir “fila anterior” con “mes anterior”.

In [3]:
periodo_expr = pl.col("PERIODO").cast(pl.Int32)
df = df.with_columns(
    (periodo_expr // 100).cast(pl.Int32).alias("ANIO"),
    (periodo_expr % 100).cast(pl.Int8).alias("MES"),
    periodo_expr.cast(pl.String).str.strptime(pl.Date, "%Y%m", strict=True).alias("FECHA_PERIODO"),
    ((periodo_expr // 100) * 12 + (periodo_expr % 100)).cast(pl.Int32).alias("MES_IDX"),
)

invalid_months = df.filter(~pl.col("MES").is_between(1, 12)).height
duplicate_cube_keys = df.group_by(CUBE_KEY).len().filter(pl.col("len") > 1).height
shared_old_keys = (
    df.group_by(["UBIGEO", "TARIFA", "PERIODO"])
    .agg(pl.col("DISTRIBUIDORA").n_unique().alias("n_distribuidoras"))
    .filter(pl.col("n_distribuidoras") > 1)
    .height
)
n_series_official = df.select(pl.struct(SERIES_KEY).n_unique()).item()

coverage_by_distributor = (
    df.group_by("DISTRIBUIDORA")
    .agg(
        pl.len().alias("filas_cubo"),
        pl.col("n_registros").sum().alias("registros_fuente"),
        pl.col("PERIODO").n_unique().alias("periodos"),
        pl.col("PERIODO").min().alias("periodo_min"),
        pl.col("PERIODO").max().alias("periodo_max"),
        pl.col("UBIGEO").n_unique().alias("ubigeos"),
        pl.col("consumo_total_mwh").sum().alias("consumo_total_mwh"),
    )
    .sort("DISTRIBUIDORA")
)
observations_by_period = df.group_by("PERIODO").len(name="observaciones").sort("PERIODO")

assert invalid_months == 0, "Se encontraron meses inválidos en PERIODO."
assert duplicate_cube_keys == 0, "La clave oficial del cubo corregido contiene duplicados."
assert set(df["DISTRIBUIDORA"].unique().to_list()) == {"ENOSA", "HIDRANDINA"}
assert int(df["n_registros"].sum()) == 51_441_180

print(f"Series oficiales DISTRIBUIDORA × UBIGEO × TARIFA: {n_series_official:,}")
print(f"Periodos con mes inválido: {invalid_months}")
print(f"Duplicados de la clave oficial: {duplicate_cube_keys}")
print(f"Claves UBIGEO × TARIFA × PERIODO compartidas, ahora separadas correctamente: {shared_old_keys}")
print("Cobertura por distribuidora:")
display(coverage_by_distributor)
print("Número de observaciones por periodo:")
display(observations_by_period)

Series oficiales DISTRIBUIDORA × UBIGEO × TARIFA: 2,167
Periodos con mes inválido: 0
Duplicados de la clave oficial: 0
Claves UBIGEO × TARIFA × PERIODO compartidas, ahora separadas correctamente: 809
Cobertura por distribuidora:


DISTRIBUIDORA,filas_cubo,registros_fuente,periodos,periodo_min,periodo_max,ubigeos,consumo_total_mwh
str,u32,u32,u32,i32,i32,u32,f64
"""ENOSA""",16765,18518031,32,202401,202608,258,2.6080e6
"""HIDRANDINA""",28196,32923149,33,202210,202507,615,4.0454e6


Número de observaciones por periodo:


PERIODO,observaciones
i32,u32
202210,856
202212,848
202301,852
202302,853
202303,853
202304,854
202305,906
202306,888
202307,888


## 3. Clave oficial y continuidad de las series

La unidad del cubo corregido es `DISTRIBUIDORA × UBIGEO × TARIFA × PERIODO`. En consecuencia, la clave de serie temporal se fija obligatoriamente como `DISTRIBUIDORA × UBIGEO × TARIFA`; no se infiere ni se reduce según los datos.

In [4]:
temporal_coverage = (
    df.group_by(SERIES_KEY)
    .agg(
        pl.col("MES_IDX").n_unique().alias("meses_observados"),
        pl.col("MES_IDX").min().alias("mes_min_idx"),
        pl.col("MES_IDX").max().alias("mes_max_idx"),
    )
    .with_columns(
        (pl.col("mes_max_idx") - pl.col("mes_min_idx") + 1).alias("meses_rango")
    )
    .with_columns(
        (pl.col("meses_rango") - pl.col("meses_observados")).alias("meses_faltantes")
    )
)
temporal_summary = temporal_coverage.select(
    pl.len().alias("series"),
    (pl.col("meses_faltantes") == 0).sum().alias("series_consecutivas"),
    (pl.col("meses_faltantes") > 0).sum().alias("series_con_huecos"),
    pl.col("meses_faltantes").sum().alias("huecos_totales"),
)

print(f"Clave de serie fijada: {SERIES_KEY}")
print("Continuidad de las series:")
display(temporal_summary)

Clave de serie fijada: ['DISTRIBUIDORA', 'UBIGEO', 'TARIFA']
Continuidad de las series:


series,series_consecutivas,series_con_huecos,huecos_totales
u32,u32,u32,i64
2167,1484,683,1869


### Corrección metodológica aplicada

Una validación adicional encontró que el cubo inicial podía sumar ENOSA e Hidrandina en una misma observación y conservar solo una etiqueta mediante `pl.first("DISTRIBUIDORA")`. El cubo fue reconstruido desde `data/interim/` incorporando `DISTRIBUIDORA` a la agrupación. Desde este punto, todas las series y todas las salidas de Machine Learning proceden exclusivamente del cubo corregido.

## 4. Feature engineering temporal y creación del target

Los lags y el target se construyen con *self joins* sobre `MES_IDX`. Así, si falta un mes calendario, el valor queda nulo y nunca se sustituye por la observación anterior disponible.

- `lag_1`, `lag_2`, `lag_3`: consumo exactamente 1, 2 y 3 meses antes.
- `TARGET_CONSUMO_MWH`: consumo exactamente un mes después.
- `TARGET_PERIODO`: periodo real del registro futuro usado como target.

In [5]:
features = df

for offset in (1, 2, 3):
    lag_source = df.select(
        *SERIES_KEY,
        (pl.col("MES_IDX") + offset).alias("MES_IDX"),
        pl.col("PERIODO").alias(f"LAG_{offset}_PERIODO"),
        pl.col("consumo_total_mwh").alias(f"lag_{offset}"),
    )
    features = features.join(lag_source, on=SERIES_KEY + ["MES_IDX"], how="left")

target_source = df.select(
    *SERIES_KEY,
    (pl.col("MES_IDX") - 1).alias("MES_IDX"),
    pl.col("PERIODO").alias("TARGET_PERIODO"),
    pl.col("consumo_total_mwh").alias("TARGET_CONSUMO_MWH"),
)
features = features.join(target_source, on=SERIES_KEY + ["MES_IDX"], how="left")

print("Muestra de variables temporales:")
display(
    features.select(
        *SERIES_KEY,
        "PERIODO", "lag_1", "lag_2", "lag_3", "TARGET_PERIODO", "TARGET_CONSUMO_MWH",
    ).sort(SERIES_KEY + ["PERIODO"]).head(10)
)

Muestra de variables temporales:


DISTRIBUIDORA,UBIGEO,TARIFA,PERIODO,lag_1,lag_2,lag_3,TARGET_PERIODO,TARGET_CONSUMO_MWH
str,str,str,i32,f64,f64,f64,i32,f64
"""ENOSA""","""010509""","""BT5B""",202407,null,null,null,202408,0.0
"""ENOSA""","""010509""","""BT5B""",202408,0.0,null,null,202409,0.0
"""ENOSA""","""010509""","""BT5B""",202409,0.0,0.0,null,202410,0.0
"""ENOSA""","""010509""","""BT5B""",202410,0.0,0.0,0.0,202411,0.0
"""ENOSA""","""010509""","""BT5B""",202411,0.0,0.0,0.0,202412,0.0
"""ENOSA""","""010509""","""BT5B""",202412,0.0,0.0,0.0,202501,0.0
"""ENOSA""","""010509""","""BT5B""",202501,0.0,0.0,0.0,202502,0.0
"""ENOSA""","""010509""","""BT5B""",202502,0.0,0.0,0.0,202503,0.0
"""ENOSA""","""010509""","""BT5B""",202503,0.0,0.0,0.0,202504,0.0


## 5. Nulos naturales y observaciones modelables

Los nulos generados por falta de `t-1`, `t-2`, `t-3` o `t+1` son una consecuencia esperada de la construcción temporal. No se imputan. Para esta primera versión se conservan solo filas con los tres lags y el target válidos.

In [6]:
null_report = features.select(
    pl.len().alias("filas_originales"),
    pl.col("lag_1").null_count().alias("sin_lag_1"),
    pl.col("lag_2").null_count().alias("sin_lag_2"),
    pl.col("lag_3").null_count().alias("sin_lag_3"),
    pl.col("TARGET_CONSUMO_MWH").null_count().alias("sin_target"),
)

modelable = features.drop_nulls(["lag_1", "lag_2", "lag_3", "TARGET_CONSUMO_MWH"])
discarded_rows = df.height - modelable.height

display(null_report)
print(f"Filas originales:          {df.height:,}")
print(f"Filas modelables finales:  {modelable.height:,}")
print(f"Filas excluidas:           {discarded_rows:,}")

filas_originales,sin_lag_1,sin_lag_2,sin_lag_3,sin_target
u32,u32,u32,u32,u32
44961,3865,5155,6718,3865


Filas originales:          44,961
Filas modelables finales:  33,174
Filas excluidas:           11,787


## 6. Partición temporal oficial

La asignación utiliza `TARGET_PERIODO`, no `PERIODO`, y mantiene meses completos:

- **TRAIN:** targets hasta 2024-12.
- **VALIDATION:** targets de 2025-01 a 2025-03.
- **TEST:** targets de 2025-04 a 2025-07.
- **FUTURE_HOLDOUT:** targets posteriores a 2025-07.

ENOSA llega hasta 2026-08, mientras Hidrandina llega hasta 2025-07. Usar 2026 para validación o prueba dejaría representada solo a ENOSA. Por eso se eligió una ventana común en la que ambas distribuidoras aparecen en TRAIN, VALIDATION y TEST. 

In [7]:
SPLIT_ORDER = ["TRAIN", "VALIDATION", "TEST", "FUTURE_HOLDOUT"]
SPLIT_RANK = {name: index for index, name in enumerate(SPLIT_ORDER)}

modelable = modelable.with_columns(
    pl.when(pl.col("TARGET_PERIODO") <= 202412)
    .then(pl.lit("TRAIN"))
    .when(pl.col("TARGET_PERIODO").is_between(202501, 202503))
    .then(pl.lit("VALIDATION"))
    .when(pl.col("TARGET_PERIODO").is_between(202504, 202507))
    .then(pl.lit("TEST"))
    .otherwise(pl.lit("FUTURE_HOLDOUT"))
    .alias("SPLIT")
)

split_summary = (
    modelable.group_by("SPLIT")
    .agg(
        pl.len().alias("cantidad_observaciones"),
        pl.col("UBIGEO").n_unique().alias("cantidad_ubigeos"),
        pl.col("TARIFA").n_unique().alias("cantidad_tarifas"),
        pl.col("DISTRIBUIDORA").n_unique().alias("cantidad_distribuidoras"),
        pl.col("TARGET_PERIODO").min().alias("periodo_min"),
        pl.col("TARGET_PERIODO").max().alias("periodo_max"),
        pl.col("DISTRIBUIDORA").unique().sort().alias("distribuidoras"),
    )
    .with_columns(
        (100 * pl.col("cantidad_observaciones") / modelable.height).round(4).alias("porcentaje"),
        pl.col("distribuidoras").list.join("|").alias("distribuidoras"),
        pl.col("SPLIT").replace_strict(SPLIT_RANK, return_dtype=pl.Int8).alias("_orden"),
    )
    .sort("_orden")
    .drop("_orden")
)

distributors = df.select("DISTRIBUIDORA").unique().sort("DISTRIBUIDORA")
split_grid = distributors.join(pl.DataFrame({"SPLIT": SPLIT_ORDER}), how="cross")
distributor_split = (
    split_grid.join(
        modelable.group_by(["DISTRIBUIDORA", "SPLIT"]).len(name="cantidad_observaciones"),
        on=["DISTRIBUIDORA", "SPLIT"],
        how="left",
    )
    .with_columns(pl.col("cantidad_observaciones").fill_null(0))
    .with_columns(pl.col("SPLIT").replace_strict(SPLIT_RANK, return_dtype=pl.Int8).alias("_orden"))
    .sort(["DISTRIBUIDORA", "_orden"])
    .drop("_orden")
)
distributor_split_matrix = (
    distributor_split.pivot(on="SPLIT", index="DISTRIBUIDORA", values="cantidad_observaciones")
    .select("DISTRIBUIDORA", *SPLIT_ORDER)
)

print("Distribución final por split:")
display(split_summary)
print("Observaciones por distribuidora y split:")
display(distributor_split_matrix)

Distribución final por split:


SPLIT,cantidad_observaciones,cantidad_ubigeos,cantidad_tarifas,cantidad_distribuidoras,periodo_min,periodo_max,distribuidoras,porcentaje
str,u32,u32,u32,u32,i32,i32,str,f64
"""TRAIN""",19552,514,15,2,202304,202412,"""ENOSA|HIDRANDINA""",58.9377
"""VALIDATION""",2936,471,14,2,202501,202503,"""ENOSA|HIDRANDINA""",8.8503
"""TEST""",3934,476,14,2,202504,202507,"""ENOSA|HIDRANDINA""",11.8587
"""FUTURE_HOLDOUT""",6752,200,14,1,202508,202608,"""ENOSA""",20.3533


Observaciones por distribuidora y split:


DISTRIBUIDORA,TRAIN,VALIDATION,TEST,FUTURE_HOLDOUT
str,u32,u32,u32,u32
"""ENOSA""",3885,1540,2072,6752
"""HIDRANDINA""",15667,1396,1862,0


## 7. Validaciones contra fuga de información

Se comprueba el orden cronológico, la correspondencia calendario de cada lag y del target, la exclusión de importe y la ausencia de encoding o transformaciones aprendidas globalmente. Las variables categóricas permanecen como texto; sus transformaciones se ajustarán exclusivamente con TRAIN en el notebook de modelado.

In [8]:
split_limits_df = (
    modelable.group_by("SPLIT")
    .agg(
        pl.col("TARGET_PERIODO").min().alias("target_min"),
        pl.col("TARGET_PERIODO").max().alias("target_max"),
    )
)
split_limits = {
    row[0]: {"min": row[1], "max": row[2]}
    for row in split_limits_df.rows()
}

def month_index(column_name: str) -> pl.Expr:
    value = pl.col(column_name).cast(pl.Int32)
    return (value // 100) * 12 + (value % 100)

lag_checks = {}
for offset in (1, 2, 3):
    lag_checks[f"lag_{offset} corresponde exactamente a t-{offset}"] = bool(
        features.filter(pl.col(f"lag_{offset}").is_not_null())
        .select((pl.col("MES_IDX") - month_index(f"LAG_{offset}_PERIODO") == offset).all())
        .item()
    )

target_calendar_ok = bool(
    features.filter(pl.col("TARGET_PERIODO").is_not_null())
    .select((month_index("TARGET_PERIODO") - pl.col("MES_IDX") == 1).all())
    .item()
)

candidate_features = [
    "UBIGEO", "TARIFA", "DISTRIBUIDORA", "ANIO", "MES",
    "consumo_total_mwh", "lag_1", "lag_2", "lag_3",
]
learned_transformations = []  # No se ajustan encoders, escaladores ni imputadores en este notebook.

checks = {
    "clave DISTRIBUIDORA × UBIGEO × TARIFA × PERIODO única": duplicate_cube_keys == 0,
    "clave de serie incluye DISTRIBUIDORA": SERIES_KEY == ["DISTRIBUIDORA", "UBIGEO", "TARIFA"],
    "TRAIN termina antes de VALIDATION": split_limits["TRAIN"]["max"] < split_limits["VALIDATION"]["min"],
    "VALIDATION termina antes de TEST": split_limits["VALIDATION"]["max"] < split_limits["TEST"]["min"],
    **lag_checks,
    "TARGET_PERIODO corresponde exactamente a t+1": target_calendar_ok,
    "importe_total_miles no está en features candidatas": "importe_total_miles" not in candidate_features,
    "target no está en features candidatas": "TARGET_CONSUMO_MWH" not in candidate_features,
    "no se realizó encoding global": all(modelable.schema[c] == pl.String for c in ["UBIGEO", "TARIFA", "DISTRIBUIDORA"]),
    "Validation/Test/Holdout no aprendieron transformaciones": len(learned_transformations) == 0,
    "ENOSA aparece en TRAIN/VALIDATION/TEST": all(
        modelable.filter((pl.col("DISTRIBUIDORA") == "ENOSA") & (pl.col("SPLIT") == split)).height > 0
        for split in ["TRAIN", "VALIDATION", "TEST"]
    ),
    "HIDRANDINA aparece en TRAIN/VALIDATION/TEST": all(
        modelable.filter((pl.col("DISTRIBUIDORA") == "HIDRANDINA") & (pl.col("SPLIT") == split)).height > 0
        for split in ["TRAIN", "VALIDATION", "TEST"]
    ),
}

leakage_validation = pl.DataFrame(
    {"validacion": list(checks.keys()), "resultado": list(checks.values())}
)

print("Límites observados de TARGET_PERIODO:")
display(split_limits_df.with_columns(
    pl.col("SPLIT").replace_strict(SPLIT_RANK, return_dtype=pl.Int8).alias("_orden")
).sort("_orden").drop("_orden"))
print("Features candidatas para el futuro pipeline:")
print(candidate_features)
print("Resultados de validación:")
display(leakage_validation)

assert all(checks.values()), "Falló al menos una validación contra fuga de información."
print("Todas las validaciones contra fuga de información fueron satisfactorias.")

Límites observados de TARGET_PERIODO:


SPLIT,target_min,target_max
str,i32,i32
"""TRAIN""",202304,202412
"""VALIDATION""",202501,202503
"""TEST""",202504,202507
"""FUTURE_HOLDOUT""",202508,202608


Features candidatas para el futuro pipeline:
['UBIGEO', 'TARIFA', 'DISTRIBUIDORA', 'ANIO', 'MES', 'consumo_total_mwh', 'lag_1', 'lag_2', 'lag_3']
Resultados de validación:


validacion,resultado
str,bool
"""clave DISTRIBUIDORA × UBIGEO ×…",true
"""clave de serie incluye DISTRIB…",true
"""TRAIN termina antes de VALIDAT…",true
"""VALIDATION termina antes de TE…",true
"""lag_1 corresponde exactamente …",true
"""lag_2 corresponde exactamente …",true
"""lag_3 corresponde exactamente …",true
"""TARGET_PERIODO corresponde exa…",true
"""importe_total_miles no está en…",true


Todas las validaciones contra fuga de información fueron satisfactorias.


## 8. Dataset final y exportación

Se conserva el consumo observado en `t`, sus tres lags, el target de `t+1`, las variables temporales y las categorías originales sin encoding. `FECHA_PERIODO` se mantiene como auxiliar de trazabilidad. El importe queda fuera del dataset de modelado.

In [9]:
final_columns = [
    "DISTRIBUIDORA", "UBIGEO", "TARIFA", "PERIODO", "FECHA_PERIODO",
    "TARGET_PERIODO", "ANIO", "MES", "consumo_total_mwh",
    "lag_1", "lag_2", "lag_3", "TARGET_CONSUMO_MWH", "SPLIT",
]
dataset_ml = modelable.select(final_columns).sort(SERIES_KEY + ["PERIODO"])

final_duplicate_count = (
    dataset_ml.group_by(SERIES_KEY + ["PERIODO"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)
final_nulls = dataset_ml.select(
    pl.col(column).null_count().alias(column) for column in dataset_ml.columns
)

assert final_duplicate_count == 0, "Existen duplicados inesperados en la clave de serie × PERIODO."
assert final_nulls.row(0) == tuple(0 for _ in dataset_ml.columns), "El dataset final contiene nulos."

dataset_ml.write_parquet(DATASET_PATH)
split_summary.write_csv(SUMMARY_PATH)
distributor_split.write_csv(DIST_SPLIT_PATH)

print(f"Dimensiones finales: {dataset_ml.shape}")
print("Schema final:")
print(dataset_ml.schema)
print("Primeras filas:")
display(dataset_ml.head())
print("Nulos por columna:")
display(final_nulls)
print(f"Duplicados de clave_de_serie × PERIODO: {final_duplicate_count}")
print("Distribución por SPLIT:")
display(dataset_ml.group_by("SPLIT").len(name="observaciones").join(
    pl.DataFrame({"SPLIT": SPLIT_ORDER, "_orden": list(range(len(SPLIT_ORDER)))}),
    on="SPLIT",
).sort("_orden").drop("_orden"))

print("Archivos exportados:")
print(f"- {DATASET_PATH}")
print(f"- {SUMMARY_PATH}")
print(f"- {DIST_SPLIT_PATH}")

Dimensiones finales: (33174, 14)


Schema final:
Schema({'DISTRIBUIDORA': String, 'UBIGEO': String, 'TARIFA': String, 'PERIODO': Int32, 'FECHA_PERIODO': Date, 'TARGET_PERIODO': Int32, 'ANIO': Int32, 'MES': Int8, 'consumo_total_mwh': Float64, 'lag_1': Float64, 'lag_2': Float64, 'lag_3': Float64, 'TARGET_CONSUMO_MWH': Float64, 'SPLIT': String})
Primeras filas:


DISTRIBUIDORA,UBIGEO,TARIFA,PERIODO,FECHA_PERIODO,TARGET_PERIODO,ANIO,MES,consumo_total_mwh,lag_1,lag_2,lag_3,TARGET_CONSUMO_MWH,SPLIT
str,str,str,i32,date,i32,i32,i8,f64,f64,f64,f64,f64,str
"""ENOSA""","""010509""","""BT5B""",202410,2024-10-01,202411,2024,10,0.0,0.0,0.0,0.0,0.0,"""TRAIN"""
"""ENOSA""","""010509""","""BT5B""",202411,2024-11-01,202412,2024,11,0.0,0.0,0.0,0.0,0.0,"""TRAIN"""
"""ENOSA""","""010509""","""BT5B""",202412,2024-12-01,202501,2024,12,0.0,0.0,0.0,0.0,0.0,"""VALIDATION"""
"""ENOSA""","""010509""","""BT5B""",202501,2025-01-01,202502,2025,1,0.0,0.0,0.0,0.0,0.0,"""VALIDATION"""
"""ENOSA""","""010509""","""BT5B""",202502,2025-02-01,202503,2025,2,0.0,0.0,0.0,0.0,0.0,"""VALIDATION"""


Nulos por columna:


DISTRIBUIDORA,UBIGEO,TARIFA,PERIODO,FECHA_PERIODO,TARGET_PERIODO,ANIO,MES,consumo_total_mwh,lag_1,lag_2,lag_3,TARGET_CONSUMO_MWH,SPLIT
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0


Duplicados de clave_de_serie × PERIODO: 0
Distribución por SPLIT:


SPLIT,observaciones
str,u32
"""TRAIN""",19552
"""VALIDATION""",2936
"""TEST""",3934
"""FUTURE_HOLDOUT""",6752


Archivos exportados:
- C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\data\processed\dataset_ml.parquet
- C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\reports\model_outputs\data_preparation_summary.csv
- C:\Users\rafae\Downloads\Hito1_1ACC0057_4773_GRUPO_08\Hito1_1ACC0057_4773_GRUPO_08\reports\model_outputs\distribucion_split_distribuidora.csv
